In [0]:
from pyspark.sql.functions import col, sum

# -----------------------------
# Paths
# -----------------------------

base_path = "/Volumes/workspace/default/my_volume_1/medallion"

input_file  = f"{base_path}/medallion_data.csv"
bronze_path = f"{base_path}/bronze_csv"
silver_path = f"{base_path}/silver_csv"
gold_path   = f"{base_path}/gold_csv"


# -----------------------------
# Bronze Layer (Raw Ingestion)
# -----------------------------

bronze_df = spark.read.csv(
    input_file,
    header=True,
    inferSchema=True
)

bronze_df.write \
.mode("overwrite") \
.option("header","true") \
.csv(bronze_path)

print("Bronze layer created")


# -----------------------------
# Silver Layer (Clean Data)
# -----------------------------

silver_df = bronze_df.filter(col("price") > 0)

silver_df.write \
.mode("overwrite") \
.option("header","true") \
.csv(silver_path)

print("Silver layer created")


# -----------------------------
# Gold Layer (Business Metrics)
# -----------------------------

gold_df = silver_df.groupBy("product") \
.agg(sum("price").alias("total_sales"))

gold_df.write \
.mode("overwrite") \
.option("header","true") \
.csv(gold_path)

print("Gold layer created")


# -----------------------------
# Show Final Output
# -----------------------------

display(gold_df)